<a href="https://colab.research.google.com/github/satyajeetprabhu/beat-this-carnatic/blob/main/notebooks/Beat_This_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Beat This! on Carnatic Music Rhythm Dataset (CMR)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Set Up Environment

In [ ]:
import os
import wandb
import glob
import shutil

In [ ]:
# Set your API key as environment variable
os.environ['WANDB_API_KEY'] = 'your-wandb-API-key'  # Replace with your actual API key

# Login to wandb (will use the environment variable)
wandb.login()
print("✅ Successfully logged in to WandB!")

wandb: Currently logged in as: satyajeetp (satyajeetp-universitat-pompeu-fabra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Successfully logged in to WandB!


In [3]:
!pip install pandas pedalboard
!pip install pytorch_lightning
!pip install git+https://github.com/satyajeetprabhu/beat-this-carnatic.git
!pip install git+https://github.com/mir-dataset-loaders/mirdata.git
!pip install mir_eval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 74.4 MB/s eta 0:00:00
  Cloning https://github.com/satyajeetprabhu/beat-this-carnatic.git to /tmp/pip-req-build-8g8dxeek
  Running command git clone --filter=blob:none --quiet https://github.com/satyajeetprabhu/beat-this-carnatic.git /tmp/pip-req-build-8g8dxeek
  Resolved https://github.com/satyajeetprabhu/beat-this-carnatic.git to commit f0142f7321a019e20f0e7da3093cb83836ca4e94
  Preparing metadata (setup.py) ... done
  Created wheel for beat-this: filename=beat_this-0.1-py3-none-any.whl size=43294 sha256=d70883bfb7a4d31c1bf178a03c2e2d55703facf738659a95667f1d88011d6c7b
  Stored in directory: /tmp/pip-ephem-wheel-cache-577c90q_/wheels/13/d6/13/719341eac270ad2cc837dd1137b7136c9308e0a136cc1c8fe3


### Clone the Beat This Repository

Clone repo if already not present.

In [ ]:
#!git clone https://github.com/satyajeetprabhu/beat-this-carnatic.git /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic

Cloning into '/content/drive/MyDrive/Beat_This_CMRdata/beat-this-carnatic'...
remote: Enumerating objects: 1335, done.
remote: Counting objects: 100% (340/340), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 1335 (delta 279), reused 270 (delta 228), pack-reused 995 (from 2)
Receiving objects: 100% (1335/1335), 82.19 MiB | 15.72 MiB/s, done.
Resolving deltas: 100% (890/890), done.
Updating files: 100% (36/36), done.


### Create shortcut to preprocessed data

Assumes folder structure from the Preprocess.ipynb notebook.  
Otherwise, create 'Beat_This_CMR' folder in your drive and upload the data folder from the pre-preprocessing output into this folder. 

In [ ]:
!ln -s /content/drive/MyDrive/Beat_This_CMR/data /content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic/data

### Set Paths

In [ ]:
path = '/content/drive/MyDrive/Beat_This_CMR/beat-this-carnatic'
data_path = os.path.join(path, 'data')

In [ ]:
# Your CMR dataset path containing the CMR_full_dataset_1.0 folder
dataset_path = '/content/drive/MyDrive/Datasets/CMR'

## Set Up Finetuning Run

### Setting these correctly for every run is critical

Run for seeds in [42, 52, 62].    
train_fold in ['cmr-fold1', 'cmr-fold2'].   
test_fold in ['cmr-fold2', 'cmr-fold1'].  

In [ ]:
seed = 62
train_fold = 'cmr-fold2'
test_fold = 'cmr-fold1'
batch_size = 4
num_workers = 4
max_epochs = 50
experiment_name = f'bt-{train_fold}-finetune-{max_epochs}'

### Create TSV file for the train fold

In [10]:
with open(f'{data_path}/audio_paths.tsv', 'w') as f:
  f.write(f'{train_fold}\tdata/audio/{train_fold}\n')

### Create .split Files

In [ ]:
from carn_utils.split_utils import export_split_tsv

In [ ]:
%cd $path

In [13]:
fold_dict = {'cmr-fold1':1, 'cmr-fold2':2}

export_split_tsv(dataset_path=dataset_path, train_fold=fold_dict[train_fold], seed=seed, csv_path='cmr_splits.csv')

80.0kB [00:00, 85.0kB/s]                            
    the research-related use you will give to the dataset. Once the access is granted (it may take, at most, one day or two), please download 
    the dataset with the provided Zenodo link and uncompress the two zip files: CMR_full_dataset_1.0.zip and CMR_subset_1.0.zip. You don't need 
    to re-arrange or change the folder structure of these two versions, the dataloader is designed to work with the provided file organization. 
    Therefore, simply uncompress and store the datasets to a desired location, and use such location to initialize the dataset as follows: 
    
    compmusic_carnatic_rhythm = mirdata.initialize("compmusic_carnatic_rhythm", data_home="/path/to/home/folder/of/dataset").
    


## Run Finetuning

In [14]:
!python launch_scripts/train.py --name {experiment_name} --checkpoint final0 --annotation-dir {train_fold} --test-dataset {test_fold} --seed {seed} --wandb-project {experiment_name} --max-epochs {max_epochs} --batch-size {batch_size} --num-workers {num_workers} --logger wandb

Seed set to 62
Starting a new run with the following parameters:
Namespace(name='bt-cmr-fold2-finetune-50', gpu=0, force_flash_attention=False, compile=['frontend', 'transformer_blocks', 'task_heads'], n_layers=6, transformer_dim=512, frontend_dropout=0.1, transformer_dropout=0.2, lr=0.0008, weight_decay=0.01, logger='wandb', wandb_project='bt-cmr-fold2-finetune-50', num_workers=4, n_heads=16, fps=50, loss='shift_tolerant_weighted_bce', warmup_steps=1000, max_epochs=50, batch_size=4, accumulate_grad_batches=8, train_length=1500, dbn=False, eval_trim_beats=5, val_frequency=5, tempo_augmentation=True, pitch_augmentation=True, mask_augmentation=True, sum_head=True, partial_transformers=True, length_based_oversampling_factor=0.65, val=True, hung_data=False, fold=None, seed=62, checkpoint='final0', annotation_dir='cmr-fold2', test_dataset='cmr-fold1', use_cpu=False)
Using data directory: data
Using checkpoint directory: checkpoints
Validation set: 18 items from: cmr-fold2 Example val file p

### Backup Trained Model Checkpoint
Run the following cell after every finetuning run to backup the best checkpoint

In [ ]:
# Define the source and destination directories
source_dir = os.path.join(path, 'checkpoints')
destination_dir = os.path.join(path, '../pretrained/bt-ft')

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Find the checkpoint file
checkpoint_file = None
for file in glob.glob(os.path.join(source_dir, f'{experiment_name}*.ckpt')):
    checkpoint_file = file
    break  # Assuming only one file will match

if checkpoint_file:
    # Define the new filename
    new_filename = f'{experiment_name} S{seed}.ckpt'
    destination_path = os.path.join(destination_dir, new_filename)

    # Copy and rename the file
    shutil.copy(checkpoint_file, destination_path)
    print(f"Copied '{checkpoint_file}' to '{destination_path}'")
else:
    print(f"No checkpoint file found matching the pattern '{experiment_name}*.ckpt' in '{source_dir}'")